# 03 - Modélisation et entraînement des modèles

## Objectif

Ce notebook a pour objectif de développer des modèles de Machine Learning capables de détecter les transactions frauduleuses dans les opérations Mobile Money.

Deux algorithmes sont étudiés :

- Régression logistique : modèle de référence pour la classification binaire.
- Random Forest : modèle d'ensemble permettant de capturer des relations non linéaires.

## Méthodologie

Les modèles sont entraînés sur l'ensemble d'entraînement issu du découpage chronologique réalisé dans le notebook précédent.

L'ensemble de validation sera utilisé ultérieurement pour comparer les modèles et ajuster les seuils de décision.

L'ensemble de test restera indépendant jusqu'à l'évaluation finale.

## Variable cible

`is_fraud` :

- 0 : transaction légitime.
- 1 : transaction frauduleuse.

Importation des bibliothèques

In [11]:
import os
import joblib
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.utils.class_weight import compute_class_weight

import warnings
warnings.filterwarnings("ignore")

print("Bibliothèques importées avec succès.")

Bibliothèques importées avec succès.


Chargement des données préparées

In [12]:
# Définition du répertoire du projet
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "notebooks").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Impossible de localiser le dossier du projet.")
    PROJECT_ROOT = PROJECT_ROOT.parent

# Chargement des ensembles
DATA_DIR = PROJECT_ROOT / "data" / "processed"

train = pd.read_parquet(DATA_DIR / "train.parquet")
valid = pd.read_parquet(DATA_DIR / "valid.parquet")
test = pd.read_parquet(DATA_DIR / "test.parquet")

print("Données chargées avec succès.")

print("Entraînement :", train.shape)
print("Validation :", valid.shape)
print("Test :", test.shape)

Données chargées avec succès.
Entraînement : (209871, 10)
Validation : (45896, 10)
Test : (44216, 10)


Séparation des variables explicatives et de la cible

In [13]:
# Variables explicatives
features = [
    "transaction_type",
    "amount",
    "heure",
    "tranche_horaire",
    "type_jour",
    "orig_type",
    "dest_type"
]

# Variable cible
target = "is_fraud"

# Séparation des données
X_train = train[features].copy()
y_train = train[target].copy()

X_valid = valid[features].copy()
y_valid = valid[target].copy()

X_test = test[features].copy()
y_test = test[target].copy()

print("Séparation des variables terminée.")

print("X_train :", X_train.shape)
print("y_train :", y_train.shape)

print("\nRépartition de la cible dans l'entraînement :")
display(y_train.value_counts())

Séparation des variables terminée.
X_train : (209871, 7)
y_train : (209871,)

Répartition de la cible dans l'entraînement :


is_fraud
0    206221
1      3650
Name: count, dtype: int64

Définition du prétraitement
Avant l'entraînement, nous devons transformer les variables afin qu'elles soient exploitables par les algorithmes.

In [14]:
# Variables numériques
numeric_features = [
    "amount",
    "heure"
]

# Variables catégorielles
categorical_features = [
    "transaction_type",
    "tranche_horaire",
    "type_jour",
    "orig_type",
    "dest_type"
]

# Prétraitement des variables numériques
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Prétraitement des variables catégorielles
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Regroupement des transformations
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Préprocesseur défini avec succès.")

Préprocesseur défini avec succès.


À comprendre :

La régression logistique a besoin de variables numériques mises à l'échelle et de catégories transformées en nombres.

Le Random Forest peut fonctionner sans normalisation, mais nous conservons ici un prétraitement commun pour simplifier notre pipeline expérimental.

Le prétraitement sera ajusté uniquement sur les données d'entraînement, afin d'éviter les fuites de données.

Création du modèle de régression logistique

In [15]:
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

print("Modèle de régression logistique créé.")

Modèle de régression logistique créé.


Entraînement de la régression logistique

In [16]:
print("Entraînement de la régression logistique en cours...")

logistic_model.fit(X_train, y_train)

print("Entraînement terminé.")

Entraînement de la régression logistique en cours...
Entraînement terminé.


Création et entraînement du Random Forest

In [17]:
random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

print("Entraînement du Random Forest en cours...")

random_forest_model.fit(X_train, y_train)

print("Entraînement terminé.")

Entraînement du Random Forest en cours...
Entraînement terminé.


Vérification de l'entraînement sur les données de

In [18]:
# Prédictions de classe
logistic_predictions = logistic_model.predict(X_valid)
rf_predictions = random_forest_model.predict(X_valid)

# Probabilités de fraude
logistic_probabilities = logistic_model.predict_proba(X_valid)[:, 1]
rf_probabilities = random_forest_model.predict_proba(X_valid)[:, 1]

# Vérification
print("Nombre de prédictions - Régression logistique :", len(logistic_predictions))
print("Nombre de prédictions - Random Forest :", len(rf_predictions))

print("\nExemples de probabilités de fraude :")

display(pd.DataFrame({
    "Valeur_reelle": y_valid.iloc[:10].values,
    "Probabilite_logistique": logistic_probabilities[:10],
    "Probabilite_random_forest": rf_probabilities[:10]
}))

Nombre de prédictions - Régression logistique : 45896
Nombre de prédictions - Random Forest : 45896

Exemples de probabilités de fraude :


,Valeur_reelle,Probabilite_logistique,Probabilite_random_forest
0,0,0.000006,0.000000
1,0,0.000006,0.000000
2,0,0.260707,0.132261
3,0,0.000015,0.000000
4,0,0.000013,0.000000
5,0,0.351131,0.092365
6,0,0.259157,0.137430
7,0,0.000006,0.000000
8,0,0.000006,0.000000
9,0,0.257717,0.151160


Sauvegarde des modèles entraînés

In [19]:
# Création du dossier de sauvegarde
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Sauvegarde des modèles complets
joblib.dump(
    logistic_model,
    MODEL_DIR / "logistic_regression.pkl"
)

joblib.dump(
    random_forest_model,
    MODEL_DIR / "random_forest.pkl"
)

# Sauvegarde des probabilités de validation
validation_predictions = pd.DataFrame({
    "y_true": y_valid.values,
    "logistic_probability": logistic_probabilities,
    "random_forest_probability": rf_probabilities
})

validation_predictions.to_parquet(
    MODEL_DIR / "validation_predictions.parquet",
    index=False
)

print("Modèles sauvegardés avec succès.")
print("Dossier :", MODEL_DIR)

Modèles sauvegardés avec succès.
Dossier : f:\COURS\MASTER\BIG DATA ... Intro & Arrchitecture des donnees\PaySim\projet-paysim\models
